# Variational Quantum Eigensolver (VQE)

This notebook implements the Variational Quantum Eigensolver (VQE) algorithm for quantum chemistry and molecular simulations.

### Pipeline Outline
1. **Environment Setup & Dependencies**: Installing Qiskit, Qiskit Nature, and PySCF.
2. **Molecular Hamiltonian Formulation**: Mapping molecular structure (e.g., $H_2$, $LiH$) to qubit operators (Jordan-Wigner / Parity transformation).
3. **Ansatz Formulation**: Parameterized quantum circuits (UCCSD / RealAmplitudes / EfficientSU2).
4. **Classical Optimization**: Classical minimizers (COBYLA, SLSQP, SPSA) evaluating ground state energy.
5. **Results & Potential Energy Surfaces**: Comparing VQE eigenvalues with Exact Diagonalization / Full CI.

In [29]:
# Setup environment, clone/pull latest repository, and install pinned dependencies
import os
import sys

!git config --global user.email "ruru-devs@users.noreply.github.com"
!git config --global user.name "ruru-devs"

if not os.path.exists("/content/Variational-Quantum-Eigensolver"):
    !git clone https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git /content/Variational-Quantum-Eigensolver
else:
    !cd /content/Variational-Quantum-Eigensolver && git pull

if "/content/Variational-Quantum-Eigensolver" not in sys.path:
    sys.path.insert(0, "/content/Variational-Quantum-Eigensolver")

%cd /content/Variational-Quantum-Eigensolver
!pip install -q -r requirements.txt


Cloning into '/content/Variational-Quantum-Eigensolver'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 16 (delta 1), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 10.70 KiB | 2.68 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Variational-Quantum-Eigensolver


In [31]:
# Run pytest with PYTHONPATH=. to verify all Phase 1 requirements
!PYTHONPATH=. pytest tests/ -v


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/Variational-Quantum-Eigensolver
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 6 items                                                              

tests/test_vqe.py::test_h2_pauli_terms_count PASSED                      [ 16%]
tests/test_vqe.py::test_sparse_hamiltonian_lowest_eigenvalue_equals_fci[H2-0.735] PASSED [ 33%]
tests/test_vqe.py::test_sparse_hamiltonian_lowest_eigenvalue_equals_fci[LiH-1.6] PASSED [ 50%]
tests/test_vqe.py::test_particle_number_and_norm_conservation PASSED     [ 66%]
tests/test_vqe.py::test_variational_principle_h2_and_lih PASSED          [ 83%]
tests/test_vqe.py::test_h2_chemical_accuracy PASSED                      [100%]

============================== 6 passed in 9.40s ===============================


In [32]:
# Cell 1: H2 One Geometry (R = 0.735 A)
!cd /content/Variational-Quantum-Eigensolver && git pull
from src.sweep import run_single_geometry

res_h2 = run_single_geometry("H2", r=0.735, results_dir="/content/Variational-Quantum-Eigensolver/results")
print(f"H2 (R=0.735 A): E_VQE={res_h2['E_vqe']:.8f} Ha | E_FCI={res_h2['E_fci']:.8f} Ha | Error={res_h2['abs_error_mHa']:.4f} mHa | Iterations={res_h2['iterations']} | Wall time={res_h2['wall_s']:.2f}s")


remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 336 bytes | 336.00 KiB/s, done.
From https://github.com/ruru-devs/Variational-Quantum-Eigensolver
   a1a661b..ed0f1ed  main       -> origin/main
Updating a1a661b..ed0f1ed
Fast-forward
 tests/__init__.py | 0
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 tests/__init__.py
H2 (R=0.735 A): E_VQE=-1.13730604 Ha | E_FCI=-1.13730604 Ha | Error=0.0000 mHa | Iterations=3 | Wall time=0.15s


In [35]:
# Cell 2: LiH One Geometry (R = 1.60 A)
from src.sweep import run_single_geometry
import shutil

res_lih = run_single_geometry("LiH", r=1.60, results_dir="/content/Variational-Quantum-Eigensolver/results")
print(f"LiH (R=1.60 A): E_VQE={res_lih['E_vqe']:.8f} Ha | E_FCI={res_lih['E_fci']:.8f} Ha | Error={res_lih['abs_error_mHa']:.4f} mHa | Iterations={res_lih['iterations']} | Wall time={res_lih['wall_s']:.2f}s")

# Git commit and push from Colab
!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Phase 1: LiH one geometry" && git push origin main


LiH (R=1.60 A): E_VQE=-7.88209660 Ha | E_FCI=-7.88209660 Ha | Error=0.0000 mHa | Iterations=10 | Wall time=0.85s
[main 9f6e1d3] Phase 1: LiH one geometry
 2 files changed, 2 insertions(+)
 create mode 100644 results/lih_single_R=1.6000.csv
 create mode 100644 results/lih_single_R=1.6000.png
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 73.60 KiB | 12.27 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git
   acd120a..9f6e1d3  main -> main


In [36]:
# Cell 3: H2O One Geometry (O-H = 0.96 A, H-O-H = 104.5 deg)
from src.sweep import run_single_geometry

# For H2O, sector dim is 225, n_params is 44. Precomputed sparse matrices evaluation is highly efficient.
res_h2o = run_single_geometry("H2O", r=0.96, angle_deg=104.5, results_dir="/content/Variational-Quantum-Eigensolver/results")
print(f"H2O (OH=0.96 A, 104.5 deg): E_VQE={res_h2o['E_vqe']:.8f} Ha | E_FCI={res_h2o['E_fci']:.8f} Ha | Error={res_h2o['abs_error_mHa']:.4f} mHa | Iterations={res_h2o['iterations']} | Wall time={res_h2o['wall_s']:.2f}s")

# Git commit and push from Colab
!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Phase 1: H2O one geometry" && git push origin main


H2O (OH=0.96 A, 104.5 deg): E_VQE=-75.01297675 Ha | E_FCI=-75.01307689 Ha | Error=0.1001 mHa | Iterations=11 | Wall time=5.95s
[main 92cb8cf] Phase 1: H2O one geometry
 2 files changed, 2 insertions(+)
 create mode 100644 results/h2o_single_R=0.9600.csv
 create mode 100644 results/h2o_single_R=0.9600.png
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 63.31 KiB | 9.04 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git
   9f6e1d3..92cb8cf  main -> main


In [37]:
# Cell 4: H2 Full Sweep (R = 0.30 to 2.50 A, step 0.05)
import numpy as np
from src.sweep import run_sweep

r_sweep_h2 = [round(float(x), 4) for x in np.arange(0.30, 2.50 + 1e-5, 0.05)]
df_h2 = run_sweep("H2", r_sweep_h2, results_dir="/content/Variational-Quantum-Eigensolver/results")

max_err_h2 = df_h2["abs_error_mHa"].max()
mean_err_h2 = df_h2["abs_error_mHa"].mean()
total_wall_h2 = df_h2["wall_s"].sum()
print(f"H2 Sweep Done: {len(df_h2)} points | Mean Error={mean_err_h2:.4f} mHa | Max Error={max_err_h2:.4f} mHa | Total Time={total_wall_h2:.2f}s")

# Git commit and push from Colab
!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Phase 1: H2 full sweep" && git push origin main


H2 Sweep Done: 45 points | Mean Error=0.0000 mHa | Max Error=0.0000 mHa | Total Time=5.38s
[main fead101] Phase 1: H2 full sweep
 3 files changed, 47 insertions(+)
 create mode 100644 results/checkpoints/sweep_h2_checkpoint.json
 create mode 100644 results/h2_sweep.csv
 create mode 100644 results/h2_sweep.png
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 125.47 KiB | 9.65 MiB/s, done.
Total 7 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git
   92cb8cf..fead101  main -> main


In [38]:
# Cell 5: LiH Full Sweep (R = 0.80 to 3.50 A, step 0.10)
import numpy as np
from src.sweep import run_sweep

r_sweep_lih = [round(float(x), 4) for x in np.arange(0.80, 3.50 + 1e-5, 0.10)]
df_lih = run_sweep("LiH", r_sweep_lih, results_dir="/content/Variational-Quantum-Eigensolver/results")

max_err_lih = df_lih["abs_error_mHa"].max()
mean_err_lih = df_lih["abs_error_mHa"].mean()
total_wall_lih = df_lih["wall_s"].sum()
print(f"LiH Sweep Done: {len(df_lih)} points | Mean Error={mean_err_lih:.4f} mHa | Max Error={max_err_lih:.4f} mHa | Total Time={total_wall_lih:.2f}s")

# Git commit and push from Colab
!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Phase 1: LiH full sweep" && git push origin main


LiH Sweep Done: 28 points | Mean Error=0.0000 mHa | Max Error=0.0000 mHa | Total Time=28.42s
[main 9520a52] Phase 1: LiH full sweep
 3 files changed, 30 insertions(+)
 create mode 100644 results/checkpoints/sweep_lih_checkpoint.json
 create mode 100644 results/lih_sweep.csv
 create mode 100644 results/lih_sweep.png
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 132.99 KiB | 10.23 MiB/s, done.
Total 7 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git
   fead101..9520a52  main -> main


In [ ]:
# Sync notebook to repo and Drive
import json
from google.colab import _message

nb_data = _message.blocking_request('get_ipynb')['ipynb']
if 'metadata' not in nb_data:
    nb_data['metadata'] = {}
if 'colab' not in nb_data['metadata']:
    nb_data['metadata']['colab'] = {}
nb_data['metadata']['colab']['name'] = "Quantum-Variational-Eigensolver.ipynb"

repo_nb_path = "/content/Variational-Quantum-Eigensolver/Quantum-Variational-Eigensolver.ipynb"
with open(repo_nb_path, "w") as f:
    json.dump(nb_data, f, indent=2)

drive_nb_path = "/content/drive/MyDrive/mol-simul-VQE/Quantum-Variational-Eigensolver.ipynb"
with open(drive_nb_path, "w") as f:
    json.dump(nb_data, f, indent=2)

colab_nb_drive = "/content/drive/MyDrive/Colab Notebooks/Quantum-Variational-Eigensolver.ipynb"
with open(colab_nb_drive, "w") as f:
    json.dump(nb_data, f, indent=2)

print("Saved cleanly to repo and Drive!")
